# Example: Building a Classical Minimum-Variance Portfolio

In this example, we generate synthetic asset return data, estimate the expected return vector and covariance matrix, and solve the minimum-variance optimization problem. We'll visualize the resulting portfolio weights and explore their sensitivity to the input estimates.

> **By the end of this example, you will be able to:**
> * Generate synthetic multi-asset return data using a multivariate normal model
> * Estimate $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$ from the synthetic data
> * Solve the minimum-variance portfolio problem using JuMP
> * Visualize portfolio weights and diagnose concentration issues

## Setup, Data and Prerequisites
We begin by loading our packages and helper functions via the `Include.jl` file. This activates the local Julia environment and loads all dependencies.

In [ ]:
include("Include.jl");

___
## Task 1: Define Our Asset Universe and Generate Synthetic Data
We'll work with a universe of $N = 5$ synthetic assets, loosely inspired by a diversified portfolio: a large-cap equity, a small-cap equity, an international equity, a bond, and a commodity. We define "true" parameters for each asset and use a multivariate normal distribution to generate synthetic daily returns.

> **Why synthetic data?** Using synthetic data lets us control the ground truth — we _know_ the true $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$, so we can measure exactly how estimation error affects the optimizer. In later sessions, we'll use [JumpHMM.jl](https://github.com/varnerlab/JumpHMM.jl) to generate more realistic regime-switching synthetic paths.

In [ ]:
let
    # Define the asset universe -
    asset_names = ["LargeCap", "SmallCap", "International", "Bond", "Commodity"];
    N = length(asset_names);

    # "True" annualized expected returns (daily = annual / 252) -
    μ_annual = [0.10, 0.12, 0.08, 0.03, 0.06]; # annual returns
    μ_true = μ_annual ./ 252; # daily returns

    # "True" annualized volatilities -
    σ_annual = [0.18, 0.25, 0.22, 0.05, 0.20];
    σ_daily = σ_annual ./ sqrt(252);

    # "True" correlation matrix -
    C_true = [
        1.00  0.70  0.65  -0.10  0.15;
        0.70  1.00  0.55  -0.05  0.20;
        0.65  0.55  1.00  -0.08  0.25;
       -0.10 -0.05 -0.08   1.00 -0.05;
        0.15  0.20  0.25  -0.05  1.00
    ];

    # Build the "true" covariance matrix: Σ = diag(σ) * C * diag(σ) -
    D = diagm(σ_daily);
    Σ_true = D * C_true * D;

    # Generate synthetic daily returns (T = 504 trading days ≈ 2 years) -
    T = 504;
    dist = MvNormal(μ_true, Σ_true);
    returns_matrix = rand(dist, T)'; # T × N matrix

    # Store in a DataFrame for convenience -
    df_returns = DataFrame(returns_matrix, asset_names);

    # save to data directory for later use -
    save(joinpath(_PATH_TO_DATA, "synthetic-returns.jld2"), 
        Dict("returns" => df_returns, "asset_names" => asset_names,
             "mu_true" => μ_true, "Sigma_true" => Σ_true, "C_true" => C_true));
    
    # show the first few rows -
    first(df_returns, 5)
end

___
## Task 2: Estimate Return and Covariance from the Synthetic Data
In practice, we don't know the true $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$ — we estimate them from historical data. Let's compute the sample estimates and see how they compare to the truth.

> **What should you see?** The sample mean and covariance will be _close_ to the true values, but not exact. With 504 observations (2 years of daily data), expected return estimates will have noticeable error bars, while covariance estimates will be more stable.

In [ ]:
let
    # Load the synthetic data -
    data = load(joinpath(_PATH_TO_DATA, "synthetic-returns.jld2"));
    df_returns = data["returns"];
    asset_names = data["asset_names"];
    μ_true = data["mu_true"];
    Σ_true = data["Sigma_true"];
    N = length(asset_names);
    
    # Compute sample estimates -
    R = Matrix(df_returns); # T × N
    μ_hat = vec(mean(R, dims=1));
    Σ_hat = cov(R);

    # Compare true vs estimated expected returns (annualized) -
    comparison = DataFrame(
        "Asset" => asset_names,
        "μ_true (annual %)" => round.(μ_true .* 252 .* 100, digits=2),
        "μ_hat (annual %)" => round.(μ_hat .* 252 .* 100, digits=2),
        "Error (bps)" => round.((μ_hat .- μ_true) .* 252 .* 10000, digits=1)
    );
    
    pretty_table(comparison, tf=tf_markdown)
end

**Visualize:** Let's compare the true and estimated correlation matrices side by side.

> **What should you see?** The estimated correlations should be close to the true values, but with some noise. The diagonal is always 1.0 by definition. Off-diagonal terms will show small deviations — this is the estimation error that the optimizer will exploit.

In [ ]:
let
    # Load data -
    data = load(joinpath(_PATH_TO_DATA, "synthetic-returns.jld2"));
    df_returns = data["returns"];
    asset_names = data["asset_names"];
    C_true = data["C_true"];
    N = length(asset_names);

    # Compute estimated correlation matrix -
    R = Matrix(df_returns);
    C_hat = cor(R);

    # Plot side by side -
    p1 = heatmap(C_true, title="True Correlation", xticks=(1:N, asset_names), 
        yticks=(1:N, asset_names), clims=(-1,1), color=:RdBu, xrotation=45, size=(400,350));
    p2 = heatmap(C_hat, title="Estimated Correlation", xticks=(1:N, asset_names), 
        yticks=(1:N, asset_names), clims=(-1,1), color=:RdBu, xrotation=45, size=(400,350));
    
    plot(p1, p2, layout=(1,2), size=(850,400))
end

___
## Task 3: Solve the Minimum-Variance Problem and Test Input Sensitivity
Now we solve the classical minimum-variance problem using the _estimated_ parameters (not the true ones — that's the whole point). We'll use the `solve_minvariance` function from our `Compute.jl` helper, which formulates the problem as a quadratic program in JuMP. Then we'll test the "error maximization" property by perturbing expected returns and seeing how the weights shift.

> **What should you see?** The optimizer will produce a weight vector that minimizes estimated portfolio variance. Watch for _concentration_ — you may see one or two assets dominate the allocation, especially the Bond (which has much lower variance than the equities).

In [ ]:
let
    # Load data -
    data = load(joinpath(_PATH_TO_DATA, "synthetic-returns.jld2"));
    df_returns = data["returns"];
    asset_names = data["asset_names"];
    N = length(asset_names);

    # Compute sample estimates -
    R = Matrix(df_returns);
    μ_hat = vec(mean(R, dims=1));
    Σ_hat = cov(R);

    # Setup the allocation problem -
    # bounds: long-only, each weight between 0 and 1
    bounds = hcat(zeros(N), ones(N));

    # target return: modest — say 5% annualized (daily = 0.05/252)
    R_target = 0.05 / 252;

    # Build and solve -
    problem = build(MyPortfolioAllocationProblem;
        μ = μ_hat, Σ = Σ_hat, bounds = bounds, R = R_target);
    result = solve_minvariance(problem);

    # Display the weights -
    weights_df = DataFrame(
        "Asset" => asset_names,
        "Weight (%)" => round.(result.weights .* 100, digits=2)
    );
    
    println("Minimum-Variance Portfolio Weights (target return = 5% annual):")
    pretty_table(weights_df, tf=tf_markdown)
    
    println("\nPortfolio expected return (annual): $(round(result.expected_return * 252 * 100, digits=2))%")
    println("Portfolio volatility (annual): $(round(sqrt(result.variance) * sqrt(252) * 100, digits=2))%")

    # Save for later use -
    save(joinpath(_PATH_TO_DATA, "baseline-portfolio.jld2"),
        Dict("weights" => result.weights, "mu_hat" => μ_hat, "Sigma_hat" => Σ_hat,
             "asset_names" => asset_names, "result" => result));
end

**Visualize:** Portfolio weight allocation as a bar chart.

> **What should you see?** The Bond asset will likely dominate the allocation because of its much lower variance. This is the _concentration problem_ in action — the optimizer loves low-variance assets regardless of whether that concentration is desirable from a diversification standpoint.

In [ ]:
let
    # Load the baseline portfolio -
    data = load(joinpath(_PATH_TO_DATA, "baseline-portfolio.jld2"));
    weights = data["weights"];
    asset_names = data["asset_names"];

    # Bar chart of weights -
    bar(asset_names, weights .* 100, 
        ylabel="Weight (%)", xlabel="Asset", title="Minimum-Variance Portfolio Weights",
        legend=false, color=:steelblue, bar_width=0.6, size=(600,400),
        ylims=(0, 100))
end

**Input Sensitivity Test:** Let's test the "error maximization" property directly. We perturb the expected return estimate for LargeCap by ±100 basis points and re-solve each time.

> **What should you see?** Even modest perturbations to a single asset's expected return can shift weights significantly. This demonstrates that the optimizer is highly sensitive to the least reliable input — estimated expected returns.

In [ ]:
let
    # Load data -
    data = load(joinpath(_PATH_TO_DATA, "baseline-portfolio.jld2"));
    μ_hat = data["mu_hat"];
    Σ_hat = data["Sigma_hat"];
    asset_names = data["asset_names"];
    baseline_weights = data["weights"];
    N = length(asset_names);
    
    # Setup -
    bounds = hcat(zeros(N), ones(N));
    R_target = 0.05 / 252;
    
    # Perturb LargeCap (index 1) expected return by ±100 bps annually -
    perturbations_bps = [-100, -50, 0, 50, 100];
    results = DataFrame();

    for δ_bps in perturbations_bps
        
        # perturb -
        μ_perturbed = copy(μ_hat);
        μ_perturbed[1] += (δ_bps / 10000) / 252; # convert annual bps to daily

        # solve -
        problem = build(MyPortfolioAllocationProblem;
            μ = μ_perturbed, Σ = Σ_hat, bounds = bounds, R = R_target);
        result = solve_minvariance(problem);

        # store -
        row = DataFrame("Perturbation (bps)" => δ_bps, 
            [name => round(w * 100, digits=2) for (name, w) in zip(asset_names, result.weights)]...);
        results = vcat(results, row);
    end

    println("LargeCap expected return perturbation → weight sensitivity:")
    pretty_table(results, tf=tf_markdown)
end

___
## Summary 
In this example, we built a classical minimum-variance portfolio from synthetic data and demonstrated two key weaknesses: weight _concentration_ (the Bond dominates) and input _sensitivity_ (small changes in expected returns shift allocations significantly). These results motivate the stress-testing framework we'll apply in the next example.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.